### Working with LLM SQL Queries in Databricks

![AI_Query_Flow](./Assets/ai_query_flow.png)

### Loading CSV file into Unity Catalog Volume

In [0]:
%sql
-- Create a Unity Catalog volume named 'genai_lab' if it does not already exist.
-- Volumes are used to store files (such as CSVs) in Databricks for data processing.
CREATE VOLUME IF NOT EXISTS genai_lab

In [0]:
import requests  # Import the requests library to handle HTTP requests

# Get the name of the current Unity Catalog catalog using a Spark SQL query
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path to the Unity Catalog volume where files will be stored
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

# Create a list of filenames to download from the remote repository
files = ["restaurant_reviews.csv"]

# Loop through each file in the list
for file in files:
    # Construct the URL to download the file from the GitHub repository
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/GenAI_Fundamentals/{file}"
    # Send an HTTP GET request to download the file
    response = requests.get(url)
    # Raise an error if the download was unsuccessful
    response.raise_for_status()

    # Open a file in write-binary mode at the target Unity Catalog volume path
    with open(f"{volume_base}/{file}", "wb") as f:
        # Write the downloaded file content to the local file
        f.write(response.content)

### Creating the Bronze Layer Dataframe

In [0]:
# Read the CSV file from the Unity Catalog volume into a Spark DataFrame.
# The 'format' parameter specifies the file type as 'csv'.
# The 'header=True' option tells Spark to use the first row of the CSV as column names.
bronze_df = spark.read.load(
    f'/Volumes/{catalog_name}/default/genai_lab/restaurant_reviews.csv',
    format='csv',
    header=True
)

# Display the first 10 rows of the DataFrame in a rich, interactive table.
display(bronze_df.limit(10))

### Creating the Bronze Schema

In [0]:
%sql
-- Use the %sql magic command to indicate that this cell contains SQL code
-- Create a schema (also known as a database) named 'Bronze' in the 'workspace' catalog if it does not already exist.
-- Schemas are used to organize tables and other database objects in Databricks.
CREATE SCHEMA IF NOT EXISTS workspace.Bronze;

### Storing the Bronze Layer as a Delta Table

In [0]:
from delta.tables import *  # Import Delta Lake table functionality for advanced operations (not used directly here)
from pyspark.sql.functions import *  # Import Spark SQL functions for DataFrame transformations (not used directly here)

# Define the path in Unity Catalog volume where the Delta table will be stored
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/delta/bronze_restaurant_reviews'

# Write the DataFrame to the specified Delta table path, overwriting any existing data
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# Save the DataFrame as a managed Delta table in the 'Bronze' schema within the Data Catalog
bronze_df.write.format('delta').saveAsTable('Bronze.bronze_restaurant_reviews')

### Creating the Gold Layer by Performing Business Operations

### Loading the Gold Layer Dataframe from the Bronze Delta Table

In [0]:
# Load the Delta table named 'bronze_restaurant_reviews' from the 'Bronze' schema as a Spark DataFrame
gold_df = spark.read.format('delta').table('Bronze.bronze_restaurant_reviews')

# Display the first 10 rows of the DataFrame in an interactive table for easy viewing
display(gold_df.limit(10))

### Using LLM to Generate Sentiment Analysis and Summarization

In [0]:
# Use Spark SQL to select columns and generate AI-powered sentiment and summary for each review
gold_df = spark.sql("""
SELECT
    CustomerID,  -- Unique identifier for each customer
    CustomerName,  -- Name of the customer who wrote the review
    Products,  -- Products ordered or mentioned in the review
    Review,  -- The actual text of the customer's review

    -- Use the ai_query function to classify the sentiment of the review as Positive, Neutral, or Negative
    ai_query(
      "databricks-llama-4-maverick",  -- Name of the LLM model to use
      "Classify the customer sentiment as Positive, Neutral, or Negative. 
       Respond with only one word.: " || Review  -- Prompt for the LLM, appending the review text
    ) AS sentiment,  -- Output column for the sentiment classification

    -- Use the ai_query function to generate a short summary of the review
    ai_query(
      "databricks-llama-4-maverick",  -- Name of the LLM model to use
      "Summarize this restaurant review in one short sentence.: " || Review  -- Prompt for the LLM, appending the review text
    ) AS summary  -- Output column for the review summary

FROM workspace.Bronze.bronze_restaurant_reviews  -- Source table containing the raw restaurant reviews
 """)

# Display the first 10 rows of the resulting DataFrame in an interactive table
display(gold_df.limit(10))

### Creating the Gold Schema in Unity Catalog

In [0]:
%sql
-- Use the %sql magic command to indicate that this cell contains SQL code


-- Create a schema (also known as a database) named 'Gold' in the 'workspace' catalog if it does not already exist.
-- Schemas are used to organize tables and other database objects in Databricks.
CREATE SCHEMA IF NOT EXISTS workspace.Gold;

### Stroing the Gold Layer as a Delta Table in Unity Catalog

In [0]:
from delta.tables import *  # Import Delta Lake table functionality for advanced operations (not used directly here)
from pyspark.sql.functions import *  # Import Spark SQL functions for DataFrame transformations (not used directly here)

# Define the path in Unity Catalog volume where the Gold Delta table will be stored
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/delta/gold_restaurant_reviews'

# Write the Gold DataFrame to the specified Delta table path, overwriting any existing data
gold_df.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Save the Gold DataFrame as a managed Delta table in the 'Gold' schema within the Data Catalog
gold_df.write.format('delta').saveAsTable('Gold.restaurant_reviews')